# Group 19 - Melbourne Nightlife Pedestrian Data Story

The analysis treats nightlife venues as contextual landmarks and pedestrian sensors as the measured data source. The results should therefore be read as **club-adjacent pedestrian activity patterns**, not as direct measurements of venue attendance or a practical guide to attending specific venues.


## 0. Dedicated import and setup cell

All imports, global settings, and output paths are collected here so the rest of the notebook stays cleaner and easier to rerun. The path logic assumes the common project layout:

```text
project-root/
├── data/raw/
├── notebooks/
└── outputs/nightclubbing_story/
```


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    import folium
    from folium.plugins import HeatMapWithTime, MarkerCluster
    FOLIUM_AVAILABLE = True
except ImportError:
    FOLIUM_AVAILABLE = False

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "outputs" / "nightclubbing_story"
CHART_DIR = OUT_DIR / "charts"
HTML_DIR = OUT_DIR / "html"
MAP_DIR = OUT_DIR / "maps"
TABLE_DIR = OUT_DIR / "tables"
TEXT_DIR = OUT_DIR / "text"

for folder in [OUT_DIR, CHART_DIR, HTML_DIR, MAP_DIR, TABLE_DIR, TEXT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT.resolve())
print("Output folder:", OUT_DIR.resolve())
print("Folium available:", FOLIUM_AVAILABLE)
print("Plotly available:", PLOTLY_AVAILABLE)


## 1. Motivation

### What is the dataset?

The core dataset is the City of Melbourne pedestrian counting system. We use two official files:

1. `pedestrian-counting-system-monthly-counts-per-hour.csv`, which contains hourly pedestrian counts by sensor.
2. `pedestrian-counting-system-sensor-locations.csv`, which contains sensor metadata such as location, status, direction, latitude, and longitude.

The project also includes a manually researched table of selected Melbourne nightlife venues, ratings, review counts, and approximate nearest pedestrian sensors. This manual table is not a replacement for the official pedestrian data; it is a contextual layer used to define club-adjacent sensor areas.

### Why did we choose these datasets?

The pedestrian counter data is suitable because nightlife is both temporal and spatial. Hourly counts let us compare early evening, late night, overnight, weekday, weekend, and seasonal patterns. Sensor locations let us connect these patterns to specific areas of the city. The manual venue table gives the story a recognizable nightlife context while still keeping the measured evidence grounded in public-space pedestrian counts.

### Goal for the end user’s experience

The website and notebook are designed to guide a reader through a clear urban data story: **which club-adjacent Melbourne streets show the strongest late-night weekend pedestrian signal, and how does that signal vary by hour and season?** The intended user experience is a combination of author-guided explanation and bounded exploration. The reader should first understand the proxy method, then see the main ranking and supporting evidence, and finally be able to explore maps and interactive charts.


## 2. Basic stats - understanding the dataset

This section loads the raw data, standardizes column names, parses dates and numeric values, merges hourly counts with sensor metadata, and creates a small summary table. In the project run represented by the original notebooks, the cleaned pedestrian table contained **1,557,320 rows**, **100 sensors**, and covered **2024-05-08 to 2026-05-07**. The code below recalculates those values from the raw files so the notebook remains reproducible if the downloaded data changes.

### Data cleaning and preprocessing choices

The most important cleaning choices are:

- standardizing column names to lowercase snake case;
- parsing dates, hours, counts, latitude, and longitude into numeric/date types;
- merging hourly counts with sensor metadata using `sensor_id`;
- using the coordinates from the count file first because they can better reflect historical sensor position, then falling back to the sensor-location file;
- dropping rows that cannot support the analysis because they are missing date, hour, pedestrian count, latitude, or longitude;
- clipping negative pedestrian counts to zero, because negative pedestrian counts are not meaningful for this analysis.


In [ ]:
counts_path = RAW_DIR / "pedestrian-counting-system-monthly-counts-per-hour.csv"
loc_path = RAW_DIR / "pedestrian-counting-system-sensor-locations.csv"

missing = [str(path.relative_to(ROOT)) for path in [counts_path, loc_path] if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing raw data files: " + ", ".join(missing) +
        "\nPlace the two City of Melbourne CSV files in data/raw/ and rerun this notebook."
    )


def clean_columns(dataframe):
    # Return a copy with simple lowercase snake-case column names.
    dataframe = dataframe.copy()
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return dataframe


def require_columns(dataframe, required, dataset_name):
    # Raise a clear error if a required column is missing.
    missing_cols = [col for col in required if col not in dataframe.columns]
    if missing_cols:
        raise KeyError(f"{dataset_name} is missing required columns: {missing_cols}")


counts_raw = pd.read_csv(counts_path)
sensors_raw = pd.read_csv(loc_path)

counts = clean_columns(counts_raw).rename(columns={
    "location_id": "sensor_id",
    "sensing_date": "date",
    "hourday": "hour",
    "total_of_directions": "pedestrian_count",
    "sensor_name": "sensor_name_counts",
})

sensors = clean_columns(sensors_raw).rename(columns={
    "location_id": "sensor_id",
    "sensor_name": "sensor_name_location",
})

require_columns(
    counts,
    ["sensor_id", "date", "hour", "pedestrian_count", "sensor_name_counts", "location"],
    "Hourly pedestrian counts",
)
require_columns(sensors, ["sensor_id", "latitude", "longitude"], "Sensor locations")

counts["date"] = pd.to_datetime(counts["date"], errors="coerce")
counts["hour"] = pd.to_numeric(counts["hour"], errors="coerce")
counts["pedestrian_count"] = pd.to_numeric(counts["pedestrian_count"], errors="coerce")

# Example location string in the counts file: "-37.81295822, 144.95678789".
coords = counts["location"].astype(str).str.split(",", expand=True)
counts["latitude_from_counts"] = pd.to_numeric(coords[0].str.strip(), errors="coerce")
counts["longitude_from_counts"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")


metadata_cols = [
    "sensor_id", "sensor_description", "sensor_name_location", "installation_date", "note",
    "location_type", "status", "direction_1", "direction_2", "latitude", "longitude",
]
for col in metadata_cols:
    if col not in sensors.columns:
        sensors[col] = pd.NA

# Merge count data with sensor metadata.
df = counts.merge(sensors[metadata_cols], on="sensor_id", how="left")

# Use coordinates from counts first, then fall back to the sensor-location table.
df["lat"] = df["latitude_from_counts"].fillna(df["latitude"])
df["lon"] = df["longitude_from_counts"].fillna(df["longitude"])
df["sensor_name"] = df["sensor_name_counts"].fillna(df["sensor_name_location"])
df["sensor_label"] = df["sensor_description"].fillna(df["sensor_name"]).fillna("Sensor " + df["sensor_id"].astype(str))

# Keep rows that can support temporal and spatial analysis.
df = df.dropna(subset=["date", "hour", "pedestrian_count", "lat", "lon"]).copy()
df["sensor_id"] = df["sensor_id"].astype(int)
df["hour"] = df["hour"].astype(int)
df["pedestrian_count"] = df["pedestrian_count"].clip(lower=0)

basic_summary = pd.DataFrame({
    "metric": [
        "rows", "sensors", "start_date", "end_date", "mean_hourly_count",
        "median_hourly_count", "max_hourly_count",
    ],
    "value": [
        len(df),
        df["sensor_id"].nunique(),
        df["date"].min().date(),
        df["date"].max().date(),
        round(df["pedestrian_count"].mean(), 2),
        round(df["pedestrian_count"].median(), 2),
        int(df["pedestrian_count"].max()),
    ],
})

display(basic_summary)
display(df.head())


### Nightlife dates, late-night windows, and seasons

A key preprocessing decision is the **nightlife date**. Calendar dates are not enough for nightlife because an observation at 01:00 on Saturday often belongs socially to Friday night. We therefore subtract one day for observations from 00:00 to 05:59. This keeps Friday night activity after midnight grouped with Friday night instead of treating it as unrelated Saturday morning activity.

We use two nightlife windows:

- **Strict late night:** 22:00–02:59. This is the main comparison window used for weekend/weekday ranking.
- **Contextual nightlife:** 18:00–05:59. This is used for rhythm plots and animated maps.


In [ ]:
df["nightlife_date"] = df["date"] - pd.to_timedelta((df["hour"] < 6).astype(int), unit="D")
df["calendar_weekday_num"] = df["date"].dt.dayofweek
df["calendar_weekday_name"] = df["date"].dt.day_name()
df["nightlife_weekday_num"] = df["nightlife_date"].dt.dayofweek
df["nightlife_weekday_name"] = df["nightlife_date"].dt.day_name()

# Friday and Saturday nights are treated as weekend nightlife.
df["nightlife_day_type"] = np.where(
    df["nightlife_weekday_num"].isin([4, 5]),
    "Weekend night",
    "Weekday night",
)


def classify_period(hour):
    if 18 <= hour <= 21:
        return "Evening, 18:00-21:59"
    if hour >= 22 or hour <= 2:
        return "Late night, 22:00-02:59"
    if 3 <= hour <= 5:
        return "Overnight, 03:00-05:59"
    if 6 <= hour <= 11:
        return "Morning, 06:00-11:59"
    return "Afternoon, 12:00-17:59"


def australian_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    if month in [3, 4, 5]:
        return "Autumn"
    if month in [6, 7, 8]:
        return "Winter"
    return "Spring"


season_order = ["Summer", "Autumn", "Winter", "Spring"]
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nightlife_hour_order = [18, 19, 20, 21, 22, 23, 0, 1, 2, 3, 4, 5]
period_order = [
    "Morning, 06:00-11:59",
    "Afternoon, 12:00-17:59",
    "Evening, 18:00-21:59",
    "Late night, 22:00-02:59",
    "Overnight, 03:00-05:59",
]

df["period"] = df["hour"].apply(classify_period)
df["is_evening"] = df["period"].eq("Evening, 18:00-21:59")
df["is_late_night"] = df["period"].eq("Late night, 22:00-02:59")
df["is_overnight"] = df["period"].eq("Overnight, 03:00-05:59")
df["is_nightlife_hour"] = df["hour"].isin(nightlife_hour_order)
df["is_weekend_night"] = df["nightlife_day_type"].eq("Weekend night")
df["is_weekday_night"] = df["nightlife_day_type"].eq("Weekday night")

df["season"] = df["nightlife_date"].dt.month.apply(australian_season)
df["season_year"] = df["nightlife_date"].dt.year
# Dec 2023, Jan 2024, and Feb 2024 are all Summer 2024.
df.loc[df["nightlife_date"].dt.month == 12, "season_year"] += 1

df["season"] = pd.Categorical(df["season"], categories=season_order, ordered=True)
df["period"] = pd.Categorical(df["period"], categories=period_order, ordered=True)
df["nightlife_weekday_name"] = pd.Categorical(df["nightlife_weekday_name"], categories=weekday_order, ordered=True)

display(df.loc[df["hour"].isin([23, 0, 1, 2]), [
    "date", "hour", "nightlife_date", "nightlife_weekday_name", "nightlife_day_type", "period", "season", "season_year"
]].head(10))


### Manual club data and sensor matching

The venue table is a manually researched contextual dataset. Review count is used as a popularity proxy, rating is used as a secondary quality proxy, and the nearest sensor assignment is used as an approximate public-space activity proxy.

One cleanup step is especially important: a few venues were initially matched to two nearby sensors. To avoid double-counting the same venue in later rankings and charts, we keep one preferred proxy sensor for those venues before building summaries.


In [ ]:
club_rows = [
    {"club_name": "Melbourne Club", "rating": 4.5, "review_count": 53, "closest_sensor_ids": [18], "closest_sensor_description_manual": "Collins Place North"},
    {"club_name": "Athenaeum Club", "rating": 4.6, "review_count": 71, "closest_sensor_ids": [39], "closest_sensor_description_manual": "Alfred Place"},
    {"club_name": "Alexandra Club", "rating": 4.8, "review_count": 17, "closest_sensor_ids": [39], "closest_sensor_description_manual": "Alfred Place"},
    {"club_name": "Kelvin Club", "rating": 4.4, "review_count": 142, "closest_sensor_ids": [39], "closest_sensor_description_manual": "Alfred Place"},
    {"club_name": "The Charlton Club", "rating": 4.0, "review_count": 2455, "closest_sensor_ids": [21, 63], "closest_sensor_description_manual": "155-161 Russell Street; 231 Bourke Street"},
    {"club_name": "Sub Club Melbourne", "rating": 3.8, "review_count": 171, "closest_sensor_ids": [84], "closest_sensor_description_manual": "Elizabeth Street - Flinders Street East"},
    {"club_name": "Mendoza Social Club", "rating": 4.1, "review_count": 175, "closest_sensor_ids": [84], "closest_sensor_description_manual": "Elizabeth Street - Flinders Street East"},
    {"club_name": "Club Moda", "rating": 4.6, "review_count": 42, "closest_sensor_ids": [36], "closest_sensor_description_manual": "Queen Street West"},
    {"club_name": "Club W", "rating": 2.6, "review_count": 169, "closest_sensor_ids": [52, 56], "closest_sensor_description_manual": "Elizabeth Street - Lonsdale Street"},
    {"club_name": "Club Retro", "rating": 3.7, "review_count": 1201, "closest_sensor_ids": [52, 56], "closest_sensor_description_manual": "Elizabeth Street - Lonsdale Street"},
    {"club_name": "Emo Never Sleeps", "rating": 4.8, "review_count": 36, "closest_sensor_ids": [182], "closest_sensor_description_manual": "163 King Street"},
    {"club_name": "Mango Club", "rating": 4.5, "review_count": 910, "closest_sensor_ids": [182], "closest_sensor_description_manual": "163 King Street"},
    {"club_name": "Bar 20 Sportsbar", "rating": 4.6, "review_count": 428, "closest_sensor_ids": [182], "closest_sensor_description_manual": "163 King Street"},
    {"club_name": "Levels Melbourne", "rating": 4.2, "review_count": 776, "closest_sensor_ids": [182], "closest_sensor_description_manual": "163 King Street"},
    {"club_name": "Decibles Nightclub", "rating": 2.8, "review_count": 60, "closest_sensor_ids": [131], "closest_sensor_description_manual": "I-Hub corner of King Street and Flinders Street"},
    {"club_name": "District 1 Melbourne", "rating": 4.2, "review_count": 558, "closest_sensor_ids": [184], "closest_sensor_description_manual": "124 Elizabeth Street"},
]

df_clubs = pd.DataFrame(club_rows)
df_clubs["log_reviews"] = np.log1p(df_clubs["review_count"])
df_clubs["rating_x_log_reviews"] = df_clubs["rating"] * df_clubs["log_reviews"]

club_sensor_pairs = (
    df_clubs
    .explode("closest_sensor_ids")
    .rename(columns={"closest_sensor_ids": "sensor_id"})
    .copy()
)
club_sensor_pairs["sensor_id"] = club_sensor_pairs["sensor_id"].astype(int)
club_sensor_pairs["club_name_clean"] = club_sensor_pairs["club_name"].str.strip().str.lower()

# Keep one preferred proxy sensor for venues that had two candidate sensors.
links_to_remove = [
    ("club w", 52),
    ("club retro", 56),
    ("the charlton club", 21),
]


df_clubs.to_csv(TABLE_DIR / "manual_club_research.csv", index=False)
club_sensor_pairs.to_csv(TABLE_DIR / "club_sensor_pairs_cleaned.csv", index=False)

display(df_clubs[["club_name", "rating", "review_count", "closest_sensor_ids"]])
print("Manual club records:", df_clubs["club_name"].nunique())
print("Club-sensor pairs after cleanup:", len(club_sensor_pairs))



## 3. Data Analysis

The analysis combines public pedestrian data with the cleaned club-sensor matching table. No machine-learning model is used. This is deliberate: our project is explanatory rather than predictive. Aggregations, ratios, percentile scores, and a transparent weighted index are more appropriate for our goal.

The score is not a black-box truth about the “best” venue. It is an interpretable index of the strongest club-adjacent pedestrian signal, based on:

- weekend late-night pedestrian intensity;
- nearby review popularity;
- weekend-versus-weekday late-night lift;
- review-weighted rating;
- strongest weekend nightlife hour.


In [ ]:
def weighted_average(values, weights):
    values = pd.Series(values)
    weights = pd.Series(weights)
    if weights.sum() == 0:
        return np.nan
    return np.average(values, weights=weights)


def percentile_score(series):
    return series.rank(pct=True).fillna(0)


def short_club_label(clubs, max_clubs=2, max_chars=55):
    if pd.isna(clubs):
        return "Unknown club-adjacent area"
    club_list = [club.strip() for club in str(clubs).split(";")]
    if len(club_list) <= max_clubs:
        label = " + ".join(club_list)
    else:
        label = " + ".join(club_list[:max_clubs]) + f" + {len(club_list) - max_clubs} more"
    return label if len(label) <= max_chars else label[:max_chars] + "..."


sensor_metadata = (
    df.sort_values("date")
    .groupby("sensor_id", as_index=False)
    .agg(
        sensor_label=("sensor_label", "last"),
        sensor_name=("sensor_name", "last"),
        lat=("lat", "last"),
        lon=("lon", "last"),
        status=("status", "last"),
        location_type=("location_type", "last"),
        note=("note", "last"),
    )
)

club_sensor_ids = sorted(club_sensor_pairs["sensor_id"].unique())


missing_sensor_ids = sorted(set(club_sensor_ids) - set(sensor_metadata["sensor_id"]))
print("Missing researched sensors:" if missing_sensor_ids else "All researched club sensors were found.", missing_sensor_ids)

club_sensor_summary = (
    club_sensor_pairs
    .groupby("sensor_id")
    .apply(lambda g: pd.Series({
        "nearby_club_count": g["club_name"].nunique(),
        "nearby_clubs": "; ".join(g.sort_values("review_count", ascending=False)["club_name"]),
        "total_reviews_nearby": g["review_count"].sum(),
        "max_reviews_single_club": g["review_count"].max(),
        "mean_club_rating": g["rating"].mean(),
        "review_weighted_rating": weighted_average(g["rating"], g["review_count"]),
        "sum_log_reviews": g["log_reviews"].sum(),
        "sum_rating_x_log_reviews": g["rating_x_log_reviews"].sum(),
        "manual_sensor_description": "; ".join(g["closest_sensor_description_manual"].unique()),
    }))
    .reset_index()
    .merge(sensor_metadata, on="sensor_id", how="left")
)
club_sensor_summary.to_csv(TABLE_DIR / "club_sensor_popularity_summary.csv", index=False)

club_df = df[df["sensor_id"].isin(club_sensor_ids)].copy()
club_nightlife = club_df[club_df["hour"].isin(nightlife_hour_order)].copy()
club_weekend_nightlife = club_nightlife[club_nightlife["is_weekend_night"]].copy()
club_weekend_late = club_nightlife[club_nightlife["is_weekend_night"] & club_nightlife["is_late_night"]].copy()
club_weekday_late = club_nightlife[club_nightlife["is_weekday_night"] & club_nightlife["is_late_night"]].copy()

print("Club-adjacent rows:", len(club_df))
print("Club-adjacent nightlife rows:", len(club_nightlife))
print("Club-adjacent weekend late-night rows:", len(club_weekend_late))


In [ ]:
sensor_all_metrics = df.groupby("sensor_id", as_index=False).agg(
    all_hours_avg=("pedestrian_count", "mean"),
    all_hours_total=("pedestrian_count", "sum"),
    observed_hours=("pedestrian_count", "size"),
)

sensor_weekend_nightlife = (
    df[df["is_nightlife_hour"] & df["is_weekend_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekend_nightlife_avg=("pedestrian_count", "mean"),
        weekend_nightlife_total=("pedestrian_count", "sum"),
        weekend_nightlife_hours=("pedestrian_count", "size"),
    )
)

sensor_weekday_nightlife = (
    df[df["is_nightlife_hour"] & df["is_weekday_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekday_nightlife_avg=("pedestrian_count", "mean"),
        weekday_nightlife_total=("pedestrian_count", "sum"),
        weekday_nightlife_hours=("pedestrian_count", "size"),
    )
)

sensor_weekend_late = (
    df[df["is_late_night"] & df["is_weekend_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekend_late_avg=("pedestrian_count", "mean"),
        weekend_late_total=("pedestrian_count", "sum"),
        weekend_late_hours=("pedestrian_count", "size"),
    )
)

sensor_weekday_late = (
    df[df["is_late_night"] & df["is_weekday_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekday_late_avg=("pedestrian_count", "mean"),
        weekday_late_total=("pedestrian_count", "sum"),
        weekday_late_hours=("pedestrian_count", "size"),
    )
)

sensor_weekend_evening = (
    df[df["is_evening"] & df["is_weekend_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekend_evening_avg=("pedestrian_count", "mean"),
        weekend_evening_total=("pedestrian_count", "sum"),
        weekend_evening_hours=("pedestrian_count", "size"),
    )
)

sensor_hourly_peak = (
    df[df["is_nightlife_hour"] & df["is_weekend_night"]]
    .groupby(["sensor_id", "hour"], as_index=False)
    .agg(avg_count=("pedestrian_count", "mean"))
    .sort_values(["sensor_id", "avg_count"], ascending=[True, False])
    .groupby("sensor_id", as_index=False)
    .first()
    .rename(columns={"hour": "best_weekend_nightlife_hour", "avg_count": "best_hour_avg_count"})
)

sensor_activity = (
    sensor_metadata
    .merge(sensor_all_metrics, on="sensor_id", how="left")
    .merge(sensor_weekend_nightlife, on="sensor_id", how="left")
    .merge(sensor_weekday_nightlife, on="sensor_id", how="left")
    .merge(sensor_weekend_late, on="sensor_id", how="left")
    .merge(sensor_weekday_late, on="sensor_id", how="left")
    .merge(sensor_weekend_evening, on="sensor_id", how="left")
    .merge(sensor_hourly_peak, on="sensor_id", how="left")
)

sensor_activity["weekend_late_vs_weekday_late_lift"] = sensor_activity["weekend_late_avg"] / sensor_activity["weekday_late_avg"]
sensor_activity["weekend_nightlife_vs_weekday_nightlife_lift"] = sensor_activity["weekend_nightlife_avg"] / sensor_activity["weekday_nightlife_avg"]

sensor_season_summary = (
    club_weekend_late
    .groupby(["sensor_id", "season"], as_index=False, observed=True)
    .agg(avg_count_per_sensor_hour=("pedestrian_count", "mean"))
)

best_sensor_season = (
    sensor_season_summary
    .sort_values(["sensor_id", "avg_count_per_sensor_hour"], ascending=[True, False])
    .groupby("sensor_id", as_index=False)
    .first()
    .rename(columns={"season": "best_season"})[["sensor_id", "best_season"]]
)

story_places = (
    club_sensor_summary
    .merge(
        sensor_activity.drop(columns=["sensor_label", "sensor_name", "lat", "lon", "status", "location_type", "note"], errors="ignore"),
        on="sensor_id",
        how="left",
    )
    .merge(best_sensor_season, on="sensor_id", how="left")
)

story_places["pedestrian_intensity_score"] = percentile_score(story_places["weekend_late_avg"])
story_places["club_popularity_score"] = percentile_score(np.log1p(story_places["total_reviews_nearby"]))
story_places["weekend_lift_score"] = percentile_score(story_places["weekend_late_vs_weekday_late_lift"].replace([np.inf, -np.inf], np.nan))
story_places["rating_score"] = story_places["review_weighted_rating"] / 5
story_places["hourly_peak_score"] = percentile_score(story_places["best_hour_avg_count"])

score_weights = {
    "pedestrian_intensity_score": 0.45,
    "club_popularity_score": 0.25,
    "weekend_lift_score": 0.15,
    "rating_score": 0.10,
    "hourly_peak_score": 0.05,
}

story_places["nightclubbing_score"] = sum(story_places[col] * weight for col, weight in score_weights.items())
story_places["nightclubbing_score_100"] = 100 * story_places["nightclubbing_score"]
story_places["club_area_label"] = story_places["nearby_clubs"].apply(short_club_label)
story_places["club_area_full_label"] = story_places["nearby_clubs"]
story_places = story_places.sort_values("nightclubbing_score_100", ascending=False).reset_index(drop=True)
story_places["rank"] = np.arange(1, len(story_places) + 1)

club_label_lookup = story_places[["sensor_id", "club_area_label", "club_area_full_label", "nearby_clubs"]].copy()
story_places.to_csv(TABLE_DIR / "best_nightclubbing_places_ranked.csv", index=False)
sensor_activity.to_csv(TABLE_DIR / "sensor_activity_metrics.csv", index=False)

rank_cols = [
    "rank", "sensor_id", "sensor_label", "club_area_label", "total_reviews_nearby",
    "review_weighted_rating", "weekend_late_avg", "weekday_late_avg",
    "weekend_late_vs_weekday_late_lift", "best_weekend_nightlife_hour",
    "best_season", "nightclubbing_score_100",
]
display(story_places[rank_cols].round(2))


### What the analysis shows

The ranking table above uses the same logic as the website story. The main result is that club-adjacent locations around Elizabeth Street, Flinders Street, Bourke Street, King Street, Queen Street, and nearby central-city corridors show different combinations of pedestrian intensity, weekend lift, review popularity, and rating. The weekend-to-weekday ratio is especially important because it separates places that are generally busy from places where activity specifically increases on Friday and Saturday nights.

The result should be interpreted carefully. Pedestrian sensors count movement through public space. They do not identify whether a person is going to a specific venue, leaving public transport, walking to another activity, or passing through the area for another reason.


## 4. Genre

Using Segel and Heer’s framework, the website is best described as a **magazine-style / annotated-chart narrative visualization** with elements of an interactive slideshow and drill-down story. It is not a fully open dashboard. Instead, it stages an argument: Melbourne’s club-adjacent late-night pedestrian signal is concentrated in a small number of central corridors, and the signal changes by weekday/weekend, hour, and season.

### Visual Narrative tools from Segel and Heer Figure 7

**Visual structuring.** The website uses a strong opening claim, section-based scrolling, repeated chart cards, chapter labels, a final verdict panel, and a ranked score component. These choices give the reader a clear route through the evidence.

**Highlighting.** The story uses score bars, rankings, circle size on maps, callout statistics, and annotations to direct attention to the strongest locations and most important comparisons.

**Transition guidance.** The website moves from method to evidence to conclusion: first the proxy method, then weekend/weekday evidence, then seasonal and hourly context, then the final ranking, then exploratory maps and interactive charts. This makes the story cumulative instead of a disconnected gallery.

### Narrative Structure tools from Segel and Heer Figure 7

**Ordering.** The story follows an author-driven sequence: research question, data and method, evidence, score, conclusion, caveats.

**Interactivity.** Maps, hoverable charts, and optional HTML outputs let the reader inspect details after the main message is established.

**Messaging.** Explanatory text, subtitles, caveats, and method notes keep the interpretation explicit. This is important because the project uses proxy measures rather than direct venue attendance data.


## 5. Visualizations

The visualizations were chosen to support the story rather than simply show every available statistic. The main visualization tasks are:

1. explain where the measured sensors are;
2. show how weekend late-night activity differs from weekday late-night activity;
3. show season and hour patterns;
4. make the scoring model transparent;
5. compare online popularity with measured public-space activity.

The cells below generate the key charts and map outputs used by the project. Static charts are saved to `outputs/nightclubbing_story/charts/`, maps to `outputs/nightclubbing_story/maps/`, and optional Plotly HTML files to `outputs/nightclubbing_story/html/`.


In [ ]:
# ------------------------------------------------------------
# Beautiful map: all sensors + researched venue proxy markers
# No clustering, custom markers, styled popups,
# and The Charlton Club popup opens on load
# ------------------------------------------------------------

import math
import html
import folium

MELBOURNE_CENTER = [-37.8136, 144.9631]

all_sensors_map_df = sensor_metadata.copy()

bars_proxy_df = (
    club_sensor_pairs
    .merge(
        sensor_metadata[["sensor_id", "sensor_label", "lat", "lon"]],
        on="sensor_id",
        how="left"
    )
    .copy()
)


bars_proxy_df["club_name_clean"] = bars_proxy_df["club_name"].str.strip().str.lower()

links_to_remove = [
    ("club w", 52),
    ("club retro", 56),
    ("the charlton club", 21),
]

remove_mask = False

for club_name, sensor_id in links_to_remove:
    remove_mask = remove_mask | (
        (bars_proxy_df["club_name_clean"] == club_name) &
        (bars_proxy_df["sensor_id"] == sensor_id)
    )

removed_links = bars_proxy_df.loc[
    remove_mask,
    ["club_name", "sensor_id", "sensor_label"]
].copy()

bars_proxy_df = bars_proxy_df.loc[~remove_mask].copy()

bars_proxy_df = bars_proxy_df.drop(columns=["club_name_clean"])


# Sort so bigger / more important venues appear first in logic
bars_proxy_df = bars_proxy_df.sort_values(
    ["review_count", "club_name"],
    ascending=[False, True]
).copy()

# Create visible proxy offsets around each linked sensor
bars_proxy_df["club_index_near_sensor"] = (
    bars_proxy_df.groupby("sensor_id").cumcount()
)

bars_proxy_df["clubs_near_sensor"] = (
    bars_proxy_df.groupby("sensor_id")["club_name"].transform("count")
)

bars_proxy_df["angle"] = (
    2 * math.pi *
    bars_proxy_df["club_index_near_sensor"] /
    bars_proxy_df["clubs_near_sensor"].clip(lower=1)
)

# Visual offset only. Increase slightly if venue markers overlap too much.
proxy_radius_degrees = 0.00046

bars_proxy_df["marker_lat"] = (
    bars_proxy_df["lat"] +
    proxy_radius_degrees * np.sin(bars_proxy_df["angle"])
)

bars_proxy_df["marker_lon"] = (
    bars_proxy_df["lon"] +
    proxy_radius_degrees * np.cos(bars_proxy_df["angle"]) /
    np.cos(np.deg2rad(bars_proxy_df["lat"]))
)

researched_sensor_ids = set(bars_proxy_df["sensor_id"].dropna().astype(int))


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def esc(value):
    if pd.isna(value):
        return "Unknown"
    return html.escape(str(value))


def fmt_num(value, decimals=0):
    if pd.isna(value):
        return "Unknown"
    if decimals == 0:
        return f"{int(value):,}"
    return f"{float(value):,.{decimals}f}"


def sensor_popup_html(row):
    sensor_id = int(row["sensor_id"])
    sensor_label = esc(row["sensor_label"])
    sensor_name = esc(row.get("sensor_name", "Unknown"))
    status = esc(row.get("status", "Unknown"))
    location_type = esc(row.get("location_type", "Unknown"))

    researched_badge = (
        '<span class="popup-badge badge-blue">Linked to researched venue</span>'
        if sensor_id in researched_sensor_ids
        else '<span class="popup-badge badge-muted">General sensor</span>'
    )

    return f"""
    <div class="map-popup">
      <div class="popup-header sensor-header">
        <div class="popup-kicker">Pedestrian sensor</div>
        <div class="popup-title">Sensor #{sensor_id}</div>
        <div class="popup-subtitle">{sensor_label}</div>
      </div>

      <div class="popup-body">
        <div class="popup-badges">
          {researched_badge}
        </div>

        <div class="popup-info-grid">
          <div>
            <span>Sensor name</span>
            <strong>{sensor_name}</strong>
          </div>
          <div>
            <span>Status</span>
            <strong>{status}</strong>
          </div>
          <div>
            <span>Location type</span>
            <strong>{location_type}</strong>
          </div>
        </div>
      </div>
    </div>
    """


def venue_popup_html(row, featured=False):
    club_name = esc(row["club_name"])
    sensor_label = esc(row["sensor_label"])
    sensor_id = int(row["sensor_id"])

    rating = fmt_num(row["rating"], 1)
    reviews = fmt_num(row["review_count"], 0)

    featured_block = """
      <div class="featured-callout">
        Featured example: this popup is opened automatically when the map loads.
      </div>
    """ if featured else ""

    return f"""
    <div class="map-popup">
      <div class="popup-header venue-header">
        <div class="popup-kicker">Researched venue</div>
        <div class="popup-title">{club_name}</div>
        <div class="popup-subtitle">Linked to Sensor #{sensor_id}</div>
      </div>

      <div class="popup-body">
        {featured_block}

        <div class="popup-stat-row">
          <div class="popup-stat">
            <span>Rating</span>
            <strong>{rating}/5</strong>
          </div>
          <div class="popup-stat">
            <span>Reviews</span>
            <strong>{reviews}</strong>
          </div>
        </div>

        <div class="popup-linked-box">
          <span>Nearest pedestrian sensor used in analysis</span>
          <strong>{sensor_label}</strong>
        </div>

        <p class="popup-note">
          This orange marker is a visual proxy placed around the linked sensor,
          not an exact venue coordinate. It shows which venue belongs to which
          pedestrian sensor area.
        </p>
      </div>
    </div>
    """


# ------------------------------------------------------------
# Build map
# ------------------------------------------------------------

m_all_places = folium.Map(
    location=[-37.8136, 144.9631],
    zoom_start=10,
    tiles="CartoDB positron",
    control_scale=True
)

# ------------------------------------------------------------
# Custom CSS for markers and popups
# ------------------------------------------------------------

custom_css = """
<style>
  .sensor-dot {
    width: 15px;
    height: 15px;
    border-radius: 999px;
    background: linear-gradient(135deg, #60a5fa, #2563eb);
    border: 2px solid #ffffff;
    box-shadow:
      0 2px 8px rgba(37, 99, 235, 0.35),
      0 0 0 3px rgba(37, 99, 235, 0.16);
  }

  .sensor-dot.researched {
    width: 18px;
    height: 18px;
    background: linear-gradient(135deg, #93c5fd, #1d4ed8);
    box-shadow:
      0 3px 10px rgba(29, 78, 216, 0.42),
      0 0 0 4px rgba(37, 99, 235, 0.18);
  }

  .venue-pin {
    width: 34px;
    height: 34px;
    display: grid;
    place-items: center;
    border-radius: 50% 50% 50% 9px;
    background: linear-gradient(135deg, #fed7aa 0%, #fb923c 42%, #c2410c 100%);
    border: 2px solid #ffffff;
    box-shadow:
      0 4px 14px rgba(194, 65, 12, 0.38),
      0 0 0 4px rgba(249, 115, 22, 0.18);
    transform: rotate(-45deg);
  }

  .venue-pin span {
    transform: rotate(45deg);
    font-size: 15px;
    line-height: 1;
  }

  .venue-pin.featured {
    animation: venuePulse 1.8s ease-in-out infinite;
    box-shadow:
      0 5px 18px rgba(194, 65, 12, 0.48),
      0 0 0 5px rgba(251, 191, 36, 0.26);
  }

  @keyframes venuePulse {
    0%, 100% {
      transform: rotate(-45deg) scale(1);
    }
    50% {
      transform: rotate(-45deg) scale(1.08);
    }
  }

  .leaflet-popup-content-wrapper {
    border-radius: 18px;
    padding: 0;
    overflow: hidden;
    box-shadow: 0 18px 45px rgba(15, 23, 42, 0.24);
  }

  .leaflet-popup-content {
    margin: 0;
    width: 340px !important;
  }

  .map-popup {
    font-family: Inter, Arial, sans-serif;
    color: #1f2937;
    background: #ffffff;
  }

  .popup-header {
    padding: 16px 18px 15px;
    color: white;
  }

  .sensor-header {
    background:
      radial-gradient(circle at 90% 10%, rgba(147, 197, 253, 0.55), transparent 34%),
      linear-gradient(135deg, #1d4ed8, #2563eb);
  }

  .venue-header {
    background:
      radial-gradient(circle at 90% 10%, rgba(254, 215, 170, 0.65), transparent 34%),
      linear-gradient(135deg, #c2410c, #f97316);
  }

  .popup-kicker {
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    opacity: 0.82;
    margin-bottom: 4px;
  }

  .popup-title {
    font-family: Syne, Inter, Arial, sans-serif;
    font-size: 19px;
    line-height: 1.1;
    font-weight: 800;
    margin-bottom: 5px;
  }

  .popup-subtitle {
    font-size: 12px;
    line-height: 1.35;
    opacity: 0.9;
  }

  .popup-body {
    padding: 16px 18px 18px;
  }

  .popup-badges {
    display: flex;
    flex-wrap: wrap;
    gap: 6px;
    margin-bottom: 12px;
  }

  .popup-badge {
    display: inline-flex;
    align-items: center;
    border-radius: 999px;
    padding: 5px 9px;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 0.03em;
    text-transform: uppercase;
  }

  .badge-blue {
    background: #dbeafe;
    color: #1d4ed8;
  }

  .badge-muted {
    background: #f3f4f6;
    color: #6b7280;
  }

  .popup-info-grid {
    display: grid;
    gap: 10px;
  }

  .popup-info-grid div,
  .popup-linked-box,
  .popup-stat {
    border: 1px solid #e5e7eb;
    background: #f9fafb;
    border-radius: 12px;
    padding: 10px 11px;
  }

  .popup-info-grid span,
  .popup-linked-box span,
  .popup-stat span {
    display: block;
    color: #6b7280;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    margin-bottom: 3px;
  }

  .popup-info-grid strong,
  .popup-linked-box strong,
  .popup-stat strong {
    display: block;
    color: #111827;
    font-size: 13px;
    line-height: 1.35;
  }

  .popup-stat-row {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 9px;
    margin-bottom: 10px;
  }

  .popup-stat strong {
    font-size: 18px;
    font-family: Syne, Inter, Arial, sans-serif;
  }

  .popup-linked-box {
    margin-bottom: 11px;
  }

  .popup-note {
    margin: 0;
    color: #6b7280;
    font-size: 11px;
    line-height: 1.55;
  }

  .featured-callout {
    border-radius: 12px;
    background: #fff7ed;
    border: 1px solid #fed7aa;
    color: #9a3412;
    padding: 10px 11px;
    font-size: 11px;
    font-weight: 700;
    line-height: 1.45;
    margin-bottom: 10px;
  }

  .map-legend-card {
    position: fixed;
    top: 10px;
    left: 50px;
    z-index: 9999;
    background: white;
    padding: 14px 16px;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    font-family: Inter, Arial, sans-serif;
    font-size: 13px;
    max-width: 390px;
    box-shadow: 0 12px 32px rgba(15, 23, 42, 0.14);
    color: #1f2937;
  }

  .map-legend-card b {
    display: block;
    font-family: Syne, Inter, Arial, sans-serif;
    font-size: 15px;
    margin-bottom: 6px;
  }

  .legend-row {
    display: flex;
    align-items: center;
    gap: 8px;
    margin-top: 6px;
    color: #4b5563;
  }

  .legend-dot-blue {
    width: 11px;
    height: 11px;
    border-radius: 50%;
    background: #2563eb;
    box-shadow: 0 0 0 3px rgba(37, 99, 235, 0.14);
  }

  .legend-dot-orange {
    width: 12px;
    height: 12px;
    border-radius: 50% 50% 50% 3px;
    transform: rotate(-45deg);
    background: #f97316;
    box-shadow: 0 0 0 3px rgba(249, 115, 22, 0.16);
  }

  .legend-line {
    width: 18px;
    height: 0;
    border-top: 2px dashed #9ca3af;
  }
</style>
"""

m_all_places.get_root().header.add_child(folium.Element(custom_css))

legend_html = """
<div class="map-legend-card">
  <b>Where sensors and venues are placed</b>
  <div class="legend-row"><span class="legend-dot-blue"></span> Pedestrian sensors</div>
  <div class="legend-row"><span class="legend-dot-orange"></span> Researched venue proxy markers</div>
  <div class="legend-row"><span class="legend-line"></span> Venue-to-sensor relationship</div>
</div>
"""

m_all_places.get_root().html.add_child(folium.Element(legend_html))


# ------------------------------------------------------------
# Layers
# ------------------------------------------------------------

sensor_layer = folium.FeatureGroup(name="Pedestrian sensors", show=True)
venue_layer = folium.FeatureGroup(name="Researched venue proxy markers", show=True)
link_layer = folium.FeatureGroup(name="Venue-to-sensor links", show=True)


# ------------------------------------------------------------
# Sensor markers
# ------------------------------------------------------------

for _, row in all_sensors_map_df.dropna(subset=["lat", "lon"]).iterrows():
    sensor_id = int(row["sensor_id"])
    is_researched = sensor_id in researched_sensor_ids

    marker_class = "sensor-dot researched" if is_researched else "sensor-dot"

    icon = folium.DivIcon(
        html=f'<div class="{marker_class}"></div>',
        icon_size=(20, 20),
        icon_anchor=(10, 10),
        class_name="custom-sensor-icon"
    )

    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=icon,
        popup=folium.Popup(
            sensor_popup_html(row),
            max_width=360
        ),
        tooltip=f"Sensor {sensor_id}: {row['sensor_label']}"
    ).add_to(sensor_layer)


# ------------------------------------------------------------
# Venue markers + link lines
# ------------------------------------------------------------

charlton_popup_opened = False

# Put Charlton first so the automatic popup chooses a strong example
bars_proxy_df["is_charlton"] = bars_proxy_df["club_name"].str.lower().eq("the charlton club")
bars_proxy_df = bars_proxy_df.sort_values(
    ["is_charlton", "review_count"],
    ascending=[False, False]
).copy()

for _, row in bars_proxy_df.dropna(subset=["marker_lat", "marker_lon"]).iterrows():
    is_charlton = str(row["club_name"]).lower() == "the charlton club"

    # Open only one Charlton popup when the map first loads
    open_this_popup = is_charlton and not charlton_popup_opened

    if open_this_popup:
        charlton_popup_opened = True

    pin_class = "venue-pin featured" if open_this_popup else "venue-pin"

    icon = folium.DivIcon(
        html=f'<div class="{pin_class}"><span>🍸</span></div>',
        icon_size=(36, 36),
        icon_anchor=(18, 34),
        class_name="custom-venue-icon"
    )

    popup = folium.Popup(
        venue_popup_html(row, featured=open_this_popup),
        max_width=380,
        show=open_this_popup
    )

    folium.Marker(
        location=[row["marker_lat"], row["marker_lon"]],
        icon=icon,
        popup=popup,
        tooltip=f"{row['club_name']} → Sensor {int(row['sensor_id'])}"
    ).add_to(venue_layer)

    folium.PolyLine(
        locations=[
            [row["marker_lat"], row["marker_lon"]],
            [row["lat"], row["lon"]]
        ],
        color="#9ca3af",
        weight=1.3,
        opacity=0.52,
        dash_array="5, 6",
        tooltip=f"{row['club_name']} linked to Sensor {int(row['sensor_id'])}"
    ).add_to(link_layer)


# ------------------------------------------------------------
# Add layers and controls
# ------------------------------------------------------------

link_layer.add_to(m_all_places)
sensor_layer.add_to(m_all_places)
venue_layer.add_to(m_all_places)

folium.LayerControl(collapsed=False).add_to(m_all_places)


# ------------------------------------------------------------
# Fit bounds
# ------------------------------------------------------------

bounds = []

for _, row in all_sensors_map_df.dropna(subset=["lat", "lon"]).iterrows():
    bounds.append([row["lat"], row["lon"]])

for _, row in bars_proxy_df.dropna(subset=["marker_lat", "marker_lon"]).iterrows():
    bounds.append([row["marker_lat"], row["marker_lon"]])

if bounds:
    m_all_places.fit_bounds(bounds)


# ------------------------------------------------------------
# Save map
# ------------------------------------------------------------

output_map_path = MAP_DIR / "all_sensors_and_venue_proxy_markers_map.html"
m_all_places.save(output_map_path)

print("Saved:", output_map_path)

m_all_places

In [ ]:
# ------------------------------------------------------------
# Bar chart: weekend vs weekday late-night activity using club names
# Modern version matching attached design
# ------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plot_df = story_places.dropna(
    subset=["weekend_late_avg", "weekday_late_avg", "club_area_label"]
).copy()

# Calculate uplift if it is not already available
plot_df["weekend_uplift"] = (
    plot_df["weekend_late_avg"] / plot_df["weekday_late_avg"].replace(0, np.nan)
)

# Sort so largest weekend nightlife areas appear at the top
plot_df = plot_df.sort_values("weekend_late_avg", ascending=True)

weekday_color = "#38BDF8"   # modern cyan-blue
weekend_color = "#A78BFA"   # modern purple
text_color = "#111827"
muted_text = "#6B7280"
grid_color = "#D1D5DB"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.titlesize": 20,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
})

fig_height = max(6.5, len(plot_df) * 0.62)
fig, ax = plt.subplots(figsize=(14, fig_height))

y = np.arange(len(plot_df))
bar_height = 0.34

weekday_bars = ax.barh(
    y - bar_height / 2,
    plot_df["weekday_late_avg"],
    height=bar_height,
    color=weekday_color,
    label="Weekday late night"
)

weekend_bars = ax.barh(
    y + bar_height / 2,
    plot_df["weekend_late_avg"],
    height=bar_height,
    color=weekend_color,
    label="Weekend late night"
)

max_value = plot_df[["weekday_late_avg", "weekend_late_avg"]].max().max()

def add_value_labels(bars, values):
    for bar, value in zip(bars, values):
        width = bar.get_width()
        y_pos = bar.get_y() + bar.get_height() / 2

        if width > max_value * 0.12:
            ax.text(
                width - max_value * 0.015,
                y_pos,
                f"{value:.0f}",
                va="center",
                ha="right",
                fontsize=10,
                fontweight="bold",
                color="white"
            )
        else:
            ax.text(
                width + max_value * 0.012,
                y_pos,
                f"{value:.0f}",
                va="center",
                ha="left",
                fontsize=10,
                color="#374151"
            )

add_value_labels(weekday_bars, plot_df["weekday_late_avg"])
add_value_labels(weekend_bars, plot_df["weekend_late_avg"])


uplift_x = max_value * 1.08
ax.text(
    uplift_x,
    len(plot_df) - 0.05,
    "Uplift",
    fontsize=11,
    fontweight="bold",
    color=muted_text,
    ha="left",
    va="bottom"
)

for i, uplift in enumerate(plot_df["weekend_uplift"]):
    if np.isfinite(uplift):
        ax.text(
            uplift_x,
            i,
            f"{uplift:.1f}x",
            va="center",
            ha="left",
            fontsize=11,
            fontweight="bold",
            color=text_color,
            bbox=dict(
                boxstyle="round,pad=0.25",
                facecolor="#F3F4F6",
                edgecolor="none"
            )
        )


ax.set_yticks(y)
ax.set_yticklabels(plot_df["club_area_label"])

ax.set_xlabel("Average pedestrians per sensor-hour, 22:00–02:59", color="#374151")
ax.set_ylabel("Nearby club area", color="#374151")

ax.set_title(
    "Weekend late-night foot traffic rises sharply near club areas",
    loc="left",
    pad=24,
    fontsize=21,
    fontweight="bold",
    color=text_color
)

ax.text(
    0,
    1.01,
    "Numbers inside bars show pedestrian averages; right column shows weekend / weekday uplift",
    transform=ax.transAxes,
    fontsize=12,
    color=muted_text,
    ha="left",
    va="bottom"
)

# Clean modern grid
ax.grid(True, axis="x", color=grid_color, alpha=0.45, linewidth=0.8)
ax.grid(False, axis="y")
ax.set_axisbelow(True)

# Add room for uplift labels
ax.set_xlim(0, max_value * 1.28)

# Round x-axis ticks
ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=7))
ax.xaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))

# Remove unnecessary chart borders
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_color("#E5E7EB")

ax.tick_params(axis="y", length=0, colors=text_color)
ax.tick_params(axis="x", colors=muted_text)

# Legend styled like the example
ax.legend(
    loc="lower right",
    frameon=True,
    facecolor="white",
    edgecolor="#E5E7EB",
    fontsize=11
)

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

plt.tight_layout()

plt.savefig(
    CHART_DIR / "weekday_vs_weekend_late_night_club_names.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

In [ ]:
# ------------------------------------------------------------
# Modern hourly nightlife rhythm near club areas
# ------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

hourly_profile = (
    club_nightlife
    .groupby(["nightlife_day_type", "hour"], as_index=False)
    .agg(avg_count=("pedestrian_count", "mean"))
)

hourly_profile["hour_order"] = hourly_profile["hour"].map(
    {hour: i for i, hour in enumerate(nightlife_hour_order)}
)

hourly_profile = hourly_profile.dropna(subset=["hour_order"]).copy()
hourly_profile["hour_order"] = hourly_profile["hour_order"].astype(int)


fig, ax = plt.subplots(figsize=(12.5, 6.6))

fig.patch.set_facecolor("#fbfbfd")
ax.set_facecolor("#ffffff")

colors = [
    "#7c3aed",  # purple
    "#06b6d4",  # cyan
    "#f59e0b",  # amber
    "#10b981",  # emerald
    "#ef4444",  # red
    "#2563eb",  # blue
]

day_types = list(hourly_profile["nightlife_day_type"].dropna().unique())


for i, day_type in enumerate(day_types):
    subset = (
        hourly_profile[hourly_profile["nightlife_day_type"] == day_type]
        .sort_values("hour_order")
    )

    color = colors[i % len(colors)]
    label = str(day_type).replace("_", " ").title()

    ax.plot(
        subset["hour_order"],
        subset["avg_count"],
        color=color,
        linewidth=3,
        marker="o",
        markersize=7,
        markerfacecolor="#ffffff",
        markeredgecolor=color,
        markeredgewidth=2,
        label=label,
        zorder=3
    )

    # Highlight each line's peak
    peak_row = subset.loc[subset["avg_count"].idxmax()]

    ax.scatter(
        peak_row["hour_order"],
        peak_row["avg_count"],
        s=115,
        color=color,
        edgecolor="#ffffff",
        linewidth=2,
        zorder=5
    )

    ax.annotate(
        f"{peak_row['avg_count']:,.0f}",
        xy=(peak_row["hour_order"], peak_row["avg_count"]),
        xytext=(0, 13),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
        color=color
    )


ax.set_xticks(range(len(nightlife_hour_order)))
ax.set_xticklabels(
    [f"{h:02d}:00" for h in nightlife_hour_order],
    fontsize=10
)

ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

ax.set_title(
    "Hourly Nightlife Pedestrian Rhythm",
    fontsize=18,
    fontweight="bold",
    loc="left",
    pad=18
)

ax.text(
    0,
    1.02,
    "Average pedestrians per sensor-hour near researched club areas",
    transform=ax.transAxes,
    fontsize=11,
    color="#6b7280",
    ha="left"
)

ax.set_xlabel("Nightlife hour", fontsize=11, labelpad=10, color="#374151")
ax.set_ylabel("Average pedestrians per sensor-hour", fontsize=11, labelpad=10, color="#374151")

ax.grid(axis="y", color="#e5e7eb", linewidth=1)
ax.grid(axis="x", visible=False)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.spines["left"].set_color("#e5e7eb")
ax.spines["bottom"].set_color("#e5e7eb")

ax.tick_params(axis="both", colors="#4b5563", length=0)

legend = ax.legend(
    frameon=True,
    loc="upper left",
    bbox_to_anchor=(0, 0.98),
    fontsize=10,
    title=None
)

legend.get_frame().set_facecolor("#ffffff")
legend.get_frame().set_edgecolor("#e5e7eb")
legend.get_frame().set_linewidth(1)

# Add a little breathing room
y_max = hourly_profile["avg_count"].max()
ax.set_ylim(0, y_max * 1.18)

plt.tight_layout()

plt.savefig(
    CHART_DIR / "hourly_nightlife_rhythm_club_areas_modern.png",
    dpi=220,
    bbox_inches="tight",
    facecolor=fig.get_facecolor()
)

plt.show()

In [ ]:
import numpy as np
from pathlib import Path
from IPython.display import HTML, display
from html import escape

CHART_DIR = Path(CHART_DIR) if "CHART_DIR" in globals() else Path("outputs/nightclubbing_story/charts")
TABLE_DIR = Path(TABLE_DIR) if "TABLE_DIR" in globals() else Path("outputs/nightclubbing_story/tables")

CHART_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

recommendations = (
    story_places
    .sort_values("nightclubbing_score_100", ascending=False)
    .copy()
    .reset_index(drop=True)
)

recommendations["rank"] = np.arange(1, len(recommendations) + 1)

recommendations["recommendation_title"] = (
    "#" + recommendations["rank"].astype(str) +
    " — " +
    recommendations["sensor_label"].astype(str)
)

recommendations["why_go_here"] = recommendations.apply(
    lambda row: (
        f"Near {row['nearby_clubs']}. "
        f"Weekend late-night foot traffic averages {row['weekend_late_avg']:.0f} pedestrians per sensor-hour. "
        f"The area has {int(row['total_reviews_nearby']):,} nearby club reviews "
        f"and a review-weighted rating of {row['review_weighted_rating']:.1f}/5."
    ),
    axis=1
)

recommendations["foot_traffic_component"] = (
    45 * recommendations["pedestrian_intensity_score"].fillna(0)
)

recommendations["reviews_component"] = (
    25 * recommendations["club_popularity_score"].fillna(0)
)

recommendations["weekend_lift_component"] = (
    15 * recommendations["weekend_lift_score"].fillna(0)
)

recommendations["rating_component"] = (
    10 * recommendations["rating_score"].fillna(0)
)

recommendations["hourly_profile_component"] = (
    5 * recommendations["hourly_peak_score"].fillna(0)
)

plot_df = recommendations.copy()

rows_html = ""

for _, row in plot_df.iterrows():
    rank = int(row["rank"])
    winner_class = " winner" if rank == 1 else ""

    label = escape(str(row["sensor_label"]))
    score = float(row["nightclubbing_score_100"])

    foot = float(row["foot_traffic_component"])
    reviews = float(row["reviews_component"])
    lift = float(row["weekend_lift_component"])
    rating = float(row["rating_component"])
    hour = float(row["hourly_profile_component"])

    title_text = (
        f"Foot traffic {foot:.1f}, "
        f"reviews {reviews:.1f}, "
        f"weekend lift {lift:.1f}, "
        f"rating {rating:.1f}, "
        f"hourly profile {hour:.1f}"
    )

    rows_html += f"""
      <div class="score-stack-row{winner_class}">
        <div class="stack-label">
          <span>{rank}</span>
          <b>{label}</b>
        </div>

        <div class="stack-bar" title="{escape(title_text)}">
          <i class="ped" style="width:{foot:.1f}%"></i>
          <i class="pop" style="width:{reviews:.1f}%"></i>
          <i class="uplift" style="width:{lift:.1f}%"></i>
          <i class="rating" style="width:{rating:.1f}%"></i>
          <i class="hour" style="width:{hour:.1f}%"></i>
        </div>

        <div class="stack-score">{score:.1f}</div>
      </div>
    """

html = f"""
<style>
  :root {{
    --accent: #a78bfa;
    --pink: #f472b6;
    --gold: #fbbf24;
    --cyan: #22d3ee;
    --card-bg: #ffffff;
    --card-border: #e5e7eb;
    --text-on-alt: #1e1b4b;
    --muted-on-alt: #5b5c6e;
    --font-display: "Syne", Inter, system-ui, sans-serif;
    --font-body: "Inter", system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
    --radius: 12px;
  }}

  .score-stack-wrap {{
    width: 100%;
    overflow-x: auto;
    overflow-y: visible;
    padding: 2px 0 8px 0;
  }}

  .score-stack-card {{
    width: 980px;
    min-width: 880px;
    max-width: none;
    background: var(--card-bg);
    border: 1px solid var(--card-border);
    border-radius: var(--radius);
    overflow: visible;
    box-sizing: border-box;
    padding: 0 0 10px 0;
    font-family: var(--font-body);
    color: #111827;
    box-shadow:
      0 14px 34px rgba(15, 23, 42, 0.06),
      0 1px 0 rgba(255, 255, 255, 0.8);
  }}

  .score-stack-card h3 {{
    margin: 0 0 4px 0;
    padding: 20px 24px 0;
    font-family: var(--font-display);
    font-size: 1rem;
    line-height: 1.15;
    font-weight: 700;
    letter-spacing: -0.01em;
    color: #000000;
  }}

  .score-stack-card .subtitle {{
    margin: 0;
    padding: 0 24px 14px;
    color: #6b7280;
    font-size: 0.8rem;
    line-height: 1.45;
    border-bottom: 1px solid var(--card-border);
  }}

  .stack-legend {{
    display: flex;
    flex-wrap: wrap;
    gap: 10px 18px;
    padding: 16px 24px 4px;
    margin: 0;
    color: #6b7280;
    font-size: 0.76rem;
    line-height: 1;
  }}

  .stack-legend span {{
    display: inline-flex;
    align-items: center;
    gap: 7px;
    white-space: nowrap;
  }}

  .stack-key {{
    width: 10px;
    height: 10px;
    border-radius: 2px;
    display: inline-block;
    box-shadow:
      inset 0 1px 0 rgba(255, 255, 255, 0.45),
      0 1px 2px rgba(15, 23, 42, 0.12);
  }}

  .score-stack-list {{
    padding: 14px 18px 18px;
    width: 100%;
    box-sizing: border-box;
  }}

  .score-stack-row {{
    display: grid;
    grid-template-columns: minmax(340px, 420px) minmax(260px, 1fr) 74px;
    align-items: center;
    gap: 14px;
    min-height: 46px;
    padding: 7px 8px 7px 0;
    border-bottom: 1px solid #f1f3f7;
    box-sizing: border-box;
  }}

  .score-stack-row:last-child {{
    border-bottom: 0;
  }}

  .stack-label {{
    display: flex;
    align-items: center;
    gap: 9px;
    min-width: 0;
    color: #111827;
    font-size: 0.82rem;
    font-weight: 700;
    line-height: 1.25;
  }}

  .stack-label span {{
    display: inline-grid;
    place-items: center;
    width: 24px;
    height: 24px;
    flex: 0 0 24px;
    border-radius: 50%;
    background: #f3f4f6;
    color: #111827;
    font-size: 0.72rem;
    font-weight: 800;
    box-shadow:
      inset 0 1px 0 rgba(255, 255, 255, 0.8),
      0 1px 2px rgba(15, 23, 42, 0.08);
  }}

  .stack-label b {{
    display: block;
    min-width: 0;
    overflow: visible;
    text-overflow: clip;
    white-space: normal;
    overflow-wrap: anywhere;
    word-break: normal;
    font-size: 0.82rem;
    font-weight: 700;
    letter-spacing: -0.01em;
  }}

  .score-stack-row.winner .stack-label,
  .score-stack-row.winner .stack-score {{
    color: #854d0e;
  }}

  .score-stack-row.winner .stack-label span {{
    background: rgba(251, 191, 36, 0.26);
    color: #854d0e;
  }}

  .stack-bar {{
    position: relative;
    display: flex;
    height: 18px;
    min-width: 0;
    overflow: hidden;
    border-radius: 999px;
    background: #eef2f7;
    box-shadow:
      inset 0 0 0 1px rgba(17, 24, 39, 0.05),
      inset 0 1px 2px rgba(17, 24, 39, 0.08),
      0 1px 2px rgba(15, 23, 42, 0.06);
  }}

  .stack-bar::before {{
    content: "";
    position: absolute;
    left: 1px;
    right: 1px;
    top: 1px;
    height: 45%;
    z-index: 2;
    border-radius: 999px 999px 40px 40px;
    pointer-events: none;
    mix-blend-mode: screen;
  }}

  .stack-bar::after {{
    content: "";
    position: absolute;
    inset: 0;
    z-index: 3;
    border-radius: inherit;
    box-shadow:
      inset 0 -6px 10px rgba(15, 23, 42, 0.10),
      inset 0 1px 0 rgba(255, 255, 255, 0.35);
    pointer-events: none;
  }}

  .stack-bar i {{
    position: relative;
    display: block;
    height: 100%;
    flex: 0 0 auto;
    box-shadow:
      inset 0 1px 0 rgba(255, 255, 255, 0.38),
      inset 0 -1px 0 rgba(15, 23, 42, 0.08);
  }}

  .ped,
  .stack-key.ped {{
    background:
      linear-gradient(180deg, rgba(255,255,255,0.32), rgba(255,255,255,0) 42%),
      linear-gradient(180deg, #b89dfb 0%, #8b6df0 100%);
  }}

  .pop,
  .stack-key.pop {{
    background:
      linear-gradient(180deg, rgba(255,255,255,0.34), rgba(255,255,255,0) 42%),
      linear-gradient(180deg, #f689c5 0%, #e85ba1 100%);
  }}

  .uplift,
  .stack-key.uplift {{
    background:
      linear-gradient(180deg, rgba(255,255,255,0.42), rgba(255,255,255,0) 44%),
      linear-gradient(180deg, #fbcc4d 0%, #f0a814 100%);
  }}

  .rating,
  .stack-key.rating {{
    background:
      linear-gradient(180deg, rgba(255,255,255,0.36), rgba(255,255,255,0) 42%),
      linear-gradient(180deg, #4ddef0 0%, #14b8c8 100%);
  }}

  .hour,
  .stack-key.hour {{
    background:
      linear-gradient(180deg, rgba(255,255,255,0.34), rgba(255,255,255,0) 42%),
      linear-gradient(180deg, #4eddab 0%, #1cba81 100%);
  }}

  .stack-score {{
    color: #111827;
    font-family: var(--font-display);
    font-size: 1rem;
    line-height: 1;
    font-weight: 800;
    text-align: right;
    font-variant-numeric: tabular-nums;
    letter-spacing: -0.02em;
    min-width: 64px;
    padding-right: 0;
    box-sizing: border-box;
  }}

  @media (max-width: 900px) {{
    .score-stack-card {{
      width: 900px;
      min-width: 900px;
    }}

    .score-stack-row {{
      grid-template-columns: minmax(310px, 370px) minmax(250px, 1fr) 74px;
    }}
  }}
</style>

<div class="score-stack-wrap">
  <div class="score-stack-card">
    <h3>Nightclubbing Area Rankings</h3>
    <p class="subtitle">
      Ranked score out of 100. Colored segments show each weighted component; the number at right is the final score.
    </p>

    <div class="stack-legend">
      <span><i class="stack-key ped"></i>Foot traffic</span>
      <span><i class="stack-key pop"></i>Reviews</span>
      <span><i class="stack-key uplift"></i>Weekend lift</span>
      <span><i class="stack-key rating"></i>Rating</span>
      <span><i class="stack-key hour"></i>Hourly profile</span>
    </div>

    <div class="score-stack-list">
      {rows_html}
    </div>
  </div>
</div>
"""


display(HTML(html))

html_path = CHART_DIR / "nightclubbing_area_rankings_glossy_score_stack.html"
html_path.write_text(html, encoding="utf-8")

recommendations[
    [
        "rank",
        "recommendation_title",
        "nearby_clubs",
        "total_reviews_nearby",
        "review_weighted_rating",
        "weekend_late_avg",
        "weekend_late_vs_weekday_late_lift",
        "best_weekend_nightlife_hour",
        "nightclubbing_score_100",
        "why_go_here"
    ]
].to_csv(TABLE_DIR / "website_recommendation_cards.csv", index=False)

print(f"Saved HTML chart to: {html_path}")
print(f"Saved recommendation table to: {TABLE_DIR / 'website_recommendation_cards.csv'}")

In [ ]:
# ------------------------------------------------------------
# Animated nightlife map: notebook version
# Source HTML: animated_nightlife_custom.html
# ------------------------------------------------------------

from pathlib import Path
from IPython.display import HTML, display

CHART_DIR = Path(CHART_DIR) if "CHART_DIR" in globals() else Path("outputs/nightclubbing_story/charts")
CHART_DIR.mkdir(parents=True, exist_ok=True)

animated_html = '<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="UTF-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1.0">\n  <title>Melbourne Night Build Animation</title>\n  <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/leaflet@1.9.4/dist/leaflet.css">\n  <script src="https://cdn.jsdelivr.net/npm/leaflet@1.9.4/dist/leaflet.js"></script>\n  <script src="https://cdn.jsdelivr.net/npm/leaflet.heat@0.2.0/dist/leaflet-heat.js"></script>\n  <style>\n    * { box-sizing: border-box; margin: 0; padding: 0; }\n    html, body { height: 100%; background: #06060f; font-family: -apple-system, \'Inter\', sans-serif; -webkit-font-smoothing: antialiased; }\n\n    #map { width: 100%; height: calc(100% - 90px); }\n\n    #controls {\n      height: 90px;\n      background: #0c0c20;\n      border-top: 1px solid rgba(255,255,255,0.08);\n      display: flex;\n      align-items: center;\n      padding: 0 16px;\n      gap: 12px;\n    }\n\n    #play-btn {\n      width: 40px; height: 40px;\n      border-radius: 50%;\n      background: #a78bfa;\n      border: none; cursor: pointer;\n      display: flex; align-items: center; justify-content: center;\n      flex-shrink: 0;\n      font-size: 13px; font-weight: 700;\n      color: #06060f;\n      transition: background 0.18s, transform 0.1s;\n      line-height: 1;\n    }\n    #play-btn:hover  { background: #c4b5fd; }\n    #play-btn:active { transform: scale(0.93); }\n\n    .timeline { flex: 1; display: flex; gap: 4px; align-items: center; overflow: hidden; }\n\n    .hour-btn {\n      flex: 1; min-width: 0;\n      padding: 7px 2px;\n      border: 1px solid rgba(255,255,255,0.09);\n      border-radius: 6px;\n      background: rgba(255,255,255,0.03);\n      color: rgba(255,255,255,0.4);\n      font-size: 0.66rem; cursor: pointer;\n      transition: all 0.15s; text-align: center;\n      font-family: inherit; white-space: nowrap;\n    }\n    .hour-btn:hover {\n      background: rgba(167,139,250,0.14);\n      border-color: rgba(167,139,250,0.4);\n      color: #fff;\n    }\n    .hour-btn.active {\n      background: #a78bfa;\n      border-color: #a78bfa;\n      color: #06060f; font-weight: 700;\n    }\n\n    .side-col { width: 68px; flex-shrink: 0; text-align: center; }\n    .side-label { font-size: 0.58rem; color: rgba(255,255,255,0.32); margin-bottom: 5px; letter-spacing: 0.1em; text-transform: uppercase; }\n    .bar-bg { background: rgba(255,255,255,0.07); border-radius: 4px; height: 6px; overflow: hidden; }\n    .bar-fill { height: 100%; background: linear-gradient(to right, #a78bfa, #f472b6); border-radius: 4px; transition: width 0.6s ease; }\n    .bar-val { font-size: 0.7rem; color: #a78bfa; font-weight: 600; margin-top: 5px; }\n\n    /* Custom tooltip styling injected into Leaflet */\n    .leaflet-tooltip.sensor-tip {\n      background: rgba(12,12,32,0.92);\n      border: 1px solid rgba(167,139,250,0.35);\n      border-radius: 8px;\n      color: #e2e8f0;\n      font-family: -apple-system, \'Inter\', sans-serif;\n      font-size: 0.78rem;\n      padding: 8px 12px;\n      pointer-events: none;\n      box-shadow: 0 4px 20px rgba(0,0,0,0.6);\n      white-space: nowrap;\n    }\n    .sensor-tip strong { color: #c4b5fd; display: block; margin-bottom: 3px; font-size: 0.82rem; }\n    .sensor-tip .tip-count { color: #fde68a; font-weight: 700; font-size: 1rem; }\n    .sensor-tip .tip-label { color: rgba(255,255,255,0.45); font-size: 0.7rem; }\n    .map-legend {\n      position: absolute;\n      right: 16px;\n      top: 16px;\n      z-index: 600;\n      background: rgba(12,12,32,0.88);\n      border: 1px solid rgba(167,139,250,0.35);\n      border-radius: 8px;\n      color: #e2e8f0;\n      padding: 12px 14px;\n      font-size: 0.72rem;\n      line-height: 1.55;\n      box-shadow: 0 10px 30px rgba(0,0,0,0.32);\n    }\n    .map-legend strong {\n      display: block;\n      color: #c4b5fd;\n      margin-bottom: 4px;\n      letter-spacing: 0.08em;\n      text-transform: uppercase;\n      font-size: 0.62rem;\n    }\n  </style>\n</head>\n<body>\n\n  <div id="map"></div>\n  <div class="map-legend">\n    <strong>Legend</strong>\n    Circle size = pedestrians/hr<br>\n    Color = hourly intensity\n  </div>\n  <div id="controls">\n    <button id="play-btn" aria-label="Play / Pause">&#9654;</button>\n    <div class="timeline" id="timeline"></div>\n    <div class="side-col">\n      <div class="side-label">Intensity</div>\n      <div class="bar-bg"><div class="bar-fill" id="act-bar" style="width:100%"></div></div>\n      <div class="bar-val" id="act-pct">100%</div>\n    </div>\n  </div>\n\n  <script>\n    // ============================================================\n    // DATA — 12 frames (18:00–05:00), 11 sensors each\n    // Source: extracted from folium animated heatmap notebook output\n    // ============================================================\n    const HOURS = [\'18:00\',\'19:00\',\'20:00\',\'21:00\',\'22:00\',\'23:00\',\'00:00\',\'01:00\',\'02:00\',\'03:00\',\'04:00\',\'05:00\'];\n\n    // Sensor names matched to coordinate order\n    const NAMES = [\n      \'Collins Place (North)\',\n      \'155–161 Russell Street\',\n      \'Queen St (West)\',\n      \'Alfred Place\',\n      \'Lonsdale–Elizabeth (North)\',\n      \'Elizabeth–Lonsdale (South)\',\n      \'231 Bourke St\',\n      \'Elizabeth–Flinders (East)\',\n      \'I-Hub King/Flinders\',\n      \'163 King Street\',\n      \'124 Elizabeth Street\'\n    ];\n\n    // [lat, lng, avg_pedestrians_per_sensor_hour]\n    const RAW = [\n      [[-37.81344862,144.97305353,164.91],[-37.81267313,144.96788288,995.55],[-37.81652527,144.96121062,367.96],[-37.81379749,144.96995745,129.83],[-37.81252157,144.96194010,789.87],[-37.81234775,144.96153311,925.20],[-37.81333081,144.96675571,891.08],[-37.81798049,144.96503383,2361.87],[-37.82009057,144.95758725,246.73],[-37.81627451,144.95550503,621.10],[-37.81512416,144.96371988,1816.20]],\n      [[-37.81344862,144.97305353,96.74],[-37.81267313,144.96788288,1010.88],[-37.81652527,144.96121062,318.14],[-37.81379749,144.96995745,88.00],[-37.81252157,144.96194010,769.64],[-37.81234775,144.96153311,928.94],[-37.81333081,144.96675571,847.68],[-37.81798049,144.96503383,1994.47],[-37.82009057,144.95758725,238.10],[-37.81627451,144.95550503,626.47],[-37.81512416,144.96371988,1515.69]],\n      [[-37.81344862,144.97305353,52.84],[-37.81267313,144.96788288,885.98],[-37.81652527,144.96121062,305.97],[-37.81379749,144.96995745,45.66],[-37.81252157,144.96194010,656.20],[-37.81234775,144.96153311,825.20],[-37.81333081,144.96675571,683.06],[-37.81798049,144.96503383,1745.93],[-37.82009057,144.95758725,234.98],[-37.81627451,144.95550503,598.26],[-37.81512416,144.96371988,1240.37]],\n      [[-37.81344862,144.97305353,39.12],[-37.81267313,144.96788288,762.26],[-37.81652527,144.96121062,286.14],[-37.81379749,144.96995745,45.90],[-37.81252157,144.96194010,543.16],[-37.81234775,144.96153311,705.50],[-37.81333081,144.96675571,569.29],[-37.81798049,144.96503383,1601.07],[-37.82009057,144.95758725,258.40],[-37.81627451,144.95550503,545.29],[-37.81512416,144.96371988,978.20]],\n      [[-37.81344862,144.97305353,44.81],[-37.81267313,144.96788288,711.30],[-37.81652527,144.96121062,311.76],[-37.81379749,144.96995745,38.56],[-37.81252157,144.96194010,458.06],[-37.81234775,144.96153311,525.06],[-37.81333081,144.96675571,501.83],[-37.81798049,144.96503383,1350.31],[-37.82009057,144.95758725,297.81],[-37.81627451,144.95550503,543.03],[-37.81512416,144.96371988,727.62]],\n      [[-37.81344862,144.97305353,28.00],[-37.81267313,144.96788288,548.65],[-37.81652527,144.96121062,330.72],[-37.81379749,144.96995745,28.54],[-37.81252157,144.96194010,421.76],[-37.81234775,144.96153311,352.29],[-37.81333081,144.96675571,375.68],[-37.81798049,144.96503383,829.35],[-37.82009057,144.95758725,302.93],[-37.81627451,144.95550503,503.43],[-37.81512416,144.96371988,490.07]],\n      [[-37.81344862,144.97305353,13.45],[-37.81267313,144.96788288,419.62],[-37.81652527,144.96121062,290.13],[-37.81379749,144.96995745,13.86],[-37.81252157,144.96194010,323.30],[-37.81234775,144.96153311,212.31],[-37.81333081,144.96675571,267.68],[-37.81798049,144.96503383,454.35],[-37.82009057,144.95758725,226.21],[-37.81627451,144.95550503,357.64],[-37.81512416,144.96371988,268.41]],\n      [[-37.81344862,144.97305353,6.85],[-37.81267313,144.96788288,288.32],[-37.81652527,144.96121062,258.07],[-37.81379749,144.96995745,8.32],[-37.81252157,144.96194010,222.91],[-37.81234775,144.96153311,140.68],[-37.81333081,144.96675571,193.33],[-37.81798049,144.96503383,296.60],[-37.82009057,144.95758725,176.71],[-37.81627451,144.95550503,266.50],[-37.81512416,144.96371988,197.01]],\n      [[-37.81344862,144.97305353,4.47],[-37.81267313,144.96788288,185.21],[-37.81652527,144.96121062,221.31],[-37.81379749,144.96995745,5.95],[-37.81252157,144.96194010,123.21],[-37.81234775,144.96153311,93.15],[-37.81333081,144.96675571,132.11],[-37.81798049,144.96503383,194.00],[-37.82009057,144.95758725,136.69],[-37.81627451,144.95550503,205.26],[-37.81512416,144.96371988,136.91]],\n      [[-37.81344862,144.97305353,3.22],[-37.81267313,144.96788288,140.89],[-37.81652527,144.96121062,235.96],[-37.81379749,144.96995745,4.89],[-37.81252157,144.96194010,98.29],[-37.81234775,144.96153311,74.07],[-37.81333081,144.96675571,92.21],[-37.81798049,144.96503383,128.77],[-37.82009057,144.95758725,113.62],[-37.81627451,144.95550503,179.70],[-37.81512416,144.96371988,94.40]],\n      [[-37.81344862,144.97305353,2.40],[-37.81267313,144.96788288,54.80],[-37.81652527,144.96121062,47.23],[-37.81379749,144.96995745,3.07],[-37.81252157,144.96194010,53.21],[-37.81234775,144.96153311,52.66],[-37.81333081,144.96675571,28.36],[-37.81798049,144.96503383,88.67],[-37.82009057,144.95758725,76.48],[-37.81627451,144.95550503,138.85],[-37.81512416,144.96371988,57.60]],\n      [[-37.81344862,144.97305353,2.43],[-37.81267313,144.96788288,27.26],[-37.81652527,144.96121062,24.80],[-37.81379749,144.96995745,4.26],[-37.81252157,144.96194010,31.06],[-37.81234775,144.96153311,47.55],[-37.81333081,144.96675571,16.48],[-37.81798049,144.96503383,75.82],[-37.82009057,144.95758725,52.71],[-37.81627451,144.95550503,84.21],[-37.81512416,144.96371988,55.72]]\n    ];\n\n    // Totals per hour + global max for normalization\n    const HOUR_TOTALS = RAW.map(f => f.reduce((s, p) => s + p[2], 0));\n    const MAX_TOTAL   = Math.max(...HOUR_TOTALS);\n    const GLOBAL_MAX  = Math.max(...RAW.flat().map(p => p[2]));  // 2361.87\n\n    // ============================================================\n    // COLOUR SCALE  indigo → purple → pink → orange → yellow\n    // ============================================================\n    const SCALE = [\n      [0.00, [30,  27,  75]],\n      [0.18, [124, 58,  237]],\n      [0.45, [219, 39,  119]],\n      [0.72, [249, 115, 22]],\n      [1.00, [253, 230, 138]]\n    ];\n\n    function valueToColor(norm) {\n      const t = Math.max(0, Math.min(1, norm));\n      let lo = SCALE[0], hi = SCALE[SCALE.length - 1];\n      for (let i = 1; i < SCALE.length; i++) {\n        if (t <= SCALE[i][0]) { lo = SCALE[i-1]; hi = SCALE[i]; break; }\n      }\n      const f = (t - lo[0]) / (hi[0] - lo[0]);\n      const r = Math.round(lo[1][0] + (hi[1][0] - lo[1][0]) * f);\n      const g = Math.round(lo[1][1] + (hi[1][1] - lo[1][1]) * f);\n      const b = Math.round(lo[1][2] + (hi[1][2] - lo[1][2]) * f);\n      return `rgb(${r},${g},${b})`;\n    }\n\n    // Radius using sqrt scale so small values are still visible\n    // Range: ~6px (dead quiet) → 40px (peak 2361)\n    function valueToRadius(v) {\n      return 6 + 34 * Math.sqrt(v / GLOBAL_MAX);\n    }\n\n    function fmtCount(v) {\n      return v >= 1000\n        ? (v / 1000).toFixed(1) + \'k\'\n        : Math.round(v).toLocaleString();\n    }\n\n    // ============================================================\n    // MAP\n    // ============================================================\n    const map = L.map(\'map\', {\n      center: [-37.8145, 144.963],\n      zoom: 14,\n      zoomControl: true,\n      attributionControl: true\n    });\n\n    L.tileLayer(\'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png\', {\n      subdomains: \'abcd\', maxZoom: 20,\n      attribution: \'© <a href="https://openstreetmap.org">OpenStreetMap</a> © <a href="https://carto.com">CARTO</a>\'\n    }).addTo(map);\n\n    // ---- Heatmap layer (global normalization, subtle glow) ----\n    let heatLayer = null;\n\n    function updateHeat(frameIdx) {\n      // Cap at 700 so mid-intensity sensors glow, not just the dominant one\n      const CAP = 700;\n      const pts = RAW[frameIdx].map(p => [p[0], p[1], Math.min(p[2] / CAP, 1.0)]);\n      if (heatLayer) map.removeLayer(heatLayer);\n      heatLayer = L.heatLayer(pts, {\n        radius: 48, blur: 38, max: 1.0, minOpacity: 0.0,\n        gradient: { 0.0: \'transparent\', 0.3: \'#1e1b4b\', 0.6: \'#7c3aed\', 1.0: \'#c084fc\' }\n      }).addTo(map);\n    }\n\n    // ---- Circle markers (update in-place each frame) ----\n    const circleMarkers = RAW[0].map((pt, i) => {\n      const norm = pt[2] / GLOBAL_MAX;\n      const m = L.circleMarker([pt[0], pt[1]], {\n        radius: valueToRadius(pt[2]),\n        fillColor: valueToColor(norm),\n        fillOpacity: 0.88,\n        color: \'rgba(255,255,255,0.55)\',\n        weight: 1.5\n      }).addTo(map);\n\n      m.bindTooltip(\'\', {\n        className: \'sensor-tip\',\n        sticky: false,\n        direction: \'top\',\n        offset: [0, -6]\n      });\n\n      return m;\n    });\n\n    function updateCircles(frameIdx) {\n      RAW[frameIdx].forEach((pt, i) => {\n        const norm = pt[2] / GLOBAL_MAX;\n        circleMarkers[i].setRadius(valueToRadius(pt[2]));\n        circleMarkers[i].setStyle({\n          fillColor: valueToColor(norm),\n          fillOpacity: 0.88\n        });\n        circleMarkers[i].setTooltipContent(\n          `<strong>${NAMES[i]}</strong>` +\n          `<span class="tip-count">${fmtCount(pt[2])}</span> ` +\n          `<span class="tip-label">pedestrians/hr</span>`\n        );\n      });\n    }\n\n    // ============================================================\n    // TIMELINE BUTTONS\n    // ============================================================\n    const timeline = document.getElementById(\'timeline\');\n    HOURS.forEach((h, i) => {\n      const btn = document.createElement(\'button\');\n      btn.className = \'hour-btn\' + (i === 0 ? \' active\' : \'\');\n      btn.textContent = h;\n      btn.addEventListener(\'click\', () => { if (playing) stopPlay(); goToFrame(i); });\n      timeline.appendChild(btn);\n    });\n\n    // ============================================================\n    // FRAME LOGIC\n    // ============================================================\n    let currentFrame = 0;\n    let playing = false;\n    let playInterval = null;\n\n    function goToFrame(i) {\n      currentFrame = i;\n      updateHeat(i);\n      updateCircles(i);\n\n      document.querySelectorAll(\'.hour-btn\').forEach((b, idx) => {\n        b.classList.toggle(\'active\', idx === i);\n      });\n\n      const pct = Math.round((HOUR_TOTALS[i] / MAX_TOTAL) * 100);\n      document.getElementById(\'act-bar\').style.width = pct + \'%\';\n      document.getElementById(\'act-pct\').textContent = pct + \'%\';\n    }\n\n    function startPlay() {\n      playing = true;\n      document.getElementById(\'play-btn\').innerHTML = \'&#9646;&#9646;\';\n      playInterval = setInterval(() => goToFrame((currentFrame + 1) % HOURS.length), 950);\n    }\n\n    function stopPlay() {\n      playing = false;\n      clearInterval(playInterval);\n      document.getElementById(\'play-btn\').innerHTML = \'&#9654;\';\n    }\n\n    document.getElementById(\'play-btn\').addEventListener(\'click\', () => {\n      playing ? stopPlay() : startPlay();\n    });\n\n    // ============================================================\n    // INIT\n    // ============================================================\n    goToFrame(0);\n  </script>\n</body>\n</html>\n'

animated_map_path = CHART_DIR / "animated_nightlife_custom.html"
animated_map_path.write_text(animated_html, encoding="utf-8")

print(f"Saved animated nightlife map to: {animated_map_path}")

iframe_html = f"""
<iframe
    src="{animated_map_path.as_posix()}"
    width="100%"
    height="720"
    style="border:0; border-radius:12px; overflow:hidden; background:#06060f;"
    loading="lazy">
</iframe>
"""

display(HTML(iframe_html))


In [ ]:
# ------------------------------------------------------------
# Scatter: club popularity vs weekend late-night pedestrians
# Pyplot version with labels on every datapoint
# ------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

club_sensor_pairs = (
    df_clubs
    .explode("closest_sensor_ids")
    .rename(columns={"closest_sensor_ids": "sensor_id"})
    .copy()
)

club_sensor_pairs["sensor_id"] = club_sensor_pairs["sensor_id"].astype(int)

club_sensor_summary = (
    club_sensor_pairs
    .groupby("sensor_id")
    .apply(
        lambda g: pd.Series({
            "nearby_club_count": g["club_name"].nunique(),
            "nearby_clubs": "; ".join(
                g.sort_values("review_count", ascending=False)["club_name"]
            ),
            "total_reviews_nearby": g["review_count"].sum(),
            "max_reviews_single_club": g["review_count"].max(),
            "mean_club_rating": g["rating"].mean(),
            "review_weighted_rating": weighted_average(g["rating"], g["review_count"]),
            "sum_log_reviews": g["log_reviews"].sum(),
            "sum_rating_x_log_reviews": g["rating_x_log_reviews"].sum(),
            "manual_sensor_description": "; ".join(
                g["closest_sensor_description_manual"].unique()
            ),
        })
    )
    .reset_index()
    .merge(sensor_metadata, on="sensor_id", how="left")
)

story_places = (
    club_sensor_summary
    .merge(
        sensor_activity.drop(
            columns=[
                "sensor_label",
                "sensor_name",
                "lat",
                "lon",
                "status",
                "location_type",
                "note"
            ],
            errors="ignore"
        ),
        on="sensor_id",
        how="left"
    )
)

plot_df = story_places.dropna(
    subset=["total_reviews_nearby", "weekend_late_avg"]
).copy()

# Rebuild the plotting label when this cell is run independently.
if "club_area_label" not in plot_df.columns:
    if "nearby_clubs" in plot_df.columns:
        plot_df["club_area_label"] = plot_df["nearby_clubs"].apply(short_club_label)
    else:
        plot_df["club_area_label"] = plot_df["sensor_label"].astype(str)

# ------------------------------------------------------------
# Highlight key areas
# ------------------------------------------------------------

plot_df["highlight_group"] = "Other areas"

plot_df.loc[
    plot_df["sensor_label"].str.contains("Queen St West", case=False, na=False),
    "highlight_group"
] = "Queen St West"

plot_df.loc[
    plot_df["sensor_label"].str.contains("Russell", case=False, na=False),
    "highlight_group"
] = "Russell St"

plot_df.loc[
    plot_df["sensor_label"].str.contains("Elizabeth", case=False, na=False)
    & plot_df["sensor_label"].str.contains("Flinders", case=False, na=False),
    "highlight_group"
] = "Elizabeth / Flinders"

# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------

colors = {
    "Other areas": "#a78bfa",
    "Queen St West": "#f472b6",
    "Russell St": "#fbbf24",
    "Elizabeth / Flinders": "#22d3ee",
}

edge_colors = {
    "Other areas": "#ffffff",
    "Queen St West": "#111827",
    "Russell St": "#111827",
    "Elizabeth / Flinders": "#111827",
}

sizes = {
    "Other areas": 130,
    "Queen St West": 260,
    "Russell St": 280,
    "Elizabeth / Flinders": 280,
}

# ------------------------------------------------------------
# Figure setup
# ------------------------------------------------------------

plt.figure(figsize=(14, 7.5), facecolor="#f8fafc")
plt.gca().set_facecolor("#f8fafc")

# ------------------------------------------------------------
# Plot each group separately so the legend works
# ------------------------------------------------------------

for group in ["Other areas", "Queen St West", "Elizabeth / Flinders", "Russell St"]:
    group_df = plot_df[plot_df["highlight_group"] == group]

    plt.scatter(
        group_df["total_reviews_nearby"],
        group_df["weekend_late_avg"],
        s=sizes[group],
        color=colors[group],
        edgecolor=edge_colors[group],
        linewidth=1.4,
        alpha=0.9,
        label=group,
        zorder=4 if group != "Other areas" else 3
    )

# ------------------------------------------------------------
# Trend line using log-scaled x
# ------------------------------------------------------------

x = plot_df["total_reviews_nearby"].values
y = plot_df["weekend_late_avg"].values

coef = np.polyfit(np.log10(x), y, 1)

x_line = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
y_line = coef[0] * np.log10(x_line) + coef[1]

plt.plot(
    x_line,
    y_line,
    linestyle=(0, (5, 5)),
    color="#64748b",
    linewidth=2,
    label="Overall trend",
    zorder=2
)

plt.text(
    1350,
    coef[0] * np.log10(1350) + coef[1] + 15,
    "overall trend",
    fontsize=12,
    color="#64748b"
)

# ------------------------------------------------------------
# Labels / “legends” for every datapoint
# ------------------------------------------------------------

label_offsets = {
    "Queen St West": (30, 30),
    "Russell St": (-110, 25),
    "Elizabeth / Flinders": (35, -45),
}

for _, row in plot_df.iterrows():
    label = row["club_area_label"]

    # Shorten long labels a bit
    label = label.replace("Elizabeth St / Flinders St", "Elizabeth/Flinders")
    label = label.replace("155-161 Russell Street", "Russell St")

    group = row["highlight_group"]

    if group in label_offsets:
        offset = label_offsets[group]
        fontsize = 12
        weight = "bold"
        color = "#111827"
    else:
        offset = (8, 8)
        fontsize = 8.5
        weight = "normal"
        color = "#475569"

    plt.annotate(
        label,
        xy=(row["total_reviews_nearby"], row["weekend_late_avg"]),
        xytext=offset,
        textcoords="offset points",
        fontsize=fontsize,
        fontweight=weight,
        color=color,
        ha="left",
        va="center",
        arrowprops=dict(
            arrowstyle="-",
            color="#94a3b8",
            linewidth=1,
            alpha=0.75
        ) if group in label_offsets else None,
        zorder=6
    )

# ------------------------------------------------------------
# Axes
# ------------------------------------------------------------

plt.xscale("log")
plt.xlim(35, 3200)
plt.ylim(-10, 670)

plt.xticks([50, 100, 300, 1000, 3000])

def format_x_tick(value, position):
    if value >= 1000:
        return f"{int(value / 1000)}k"
    return f"{int(value)}"

plt.gca().xaxis.set_major_formatter(FuncFormatter(format_x_tick))

plt.title(
    "Reviews help, but they do not fully explain foot traffic",
    fontsize=24,
    fontweight="bold",
    color="#111827",
    loc="left",
    pad=20
)

plt.xlabel(
    "Nearby club reviews (log scale: each step is a bigger jump in reviews)",
    fontsize=13,
    color="#111827",
    labelpad=14
)

plt.ylabel(
    "Weekend late-night pedestrians per sensor-hour",
    fontsize=13,
    color="#111827",
    labelpad=12
)

# Extra axis guide text
plt.text(
    0.02,
    -0.13,
    "fewer reviews",
    transform=plt.gca().transAxes,
    ha="left",
    fontsize=11,
    color="#64748b"
)

plt.text(
    0.98,
    -0.13,
    "more reviews",
    transform=plt.gca().transAxes,
    ha="right",
    fontsize=11,
    color="#64748b"
)

# ------------------------------------------------------------
# Grid, legend, and cleanup
# ------------------------------------------------------------

plt.grid(True, which="major", color="#cbd5e1", alpha=0.55, linewidth=1)
plt.grid(False, which="minor")


for spine in ["top", "right"]:
    plt.gca().spines[spine].set_visible(False)

plt.gca().spines["left"].set_color("#cbd5e1")
plt.gca().spines["bottom"].set_color("#cbd5e1")

plt.tick_params(axis="both", labelsize=11, colors="#334155")

plt.tight_layout()

plt.savefig(
    CHART_DIR / "club_popularity_vs_weekend_late_pedestrians_club_names.png",
    dpi=220,
    facecolor="#f8fafc",
    bbox_inches="tight"
)

plt.show()

### Why these visualizations fit the story

The **sensor score map** explains the spatial proxy method. It shows that the analysis is based on pedestrian sensors near venues, not direct counts of venue attendance. This is important before the reader sees a ranking.

The **weekend vs weekday late-night chart** is the central evidence chart. It compares the strict late-night window, 22:00–02:59, across club-adjacent areas. This is the clearest way to show where weekend nightlife changes the pedestrian pattern.

The **hourly rhythm chart** shows nightlife as a temporal process. It helps the reader see whether activity builds in the evening, peaks late, or continues into the early morning.

The **seasonal chart** adds a longer time scale. It shows whether the late-night signal is stable or varies across Australian seasons.

The **score component chart** makes the final ranking transparent. Instead of asking the reader to trust a single number, it shows how pedestrian intensity, popularity, weekend lift, rating, and hourly peak strength contribute.

The **review popularity vs foot traffic scatter plot** acts as a sanity check. It shows whether online popularity and measured public-space activity tell the same story or diverge.

The **animated heat map** is most useful as an exploratory website element because it lets readers inspect how the city changes hour by hour during the broader 18:00–05:59 nightlife window.


## 6. Discussion

Several parts of the project worked well. First, the nightlife-date logic makes the time analysis more faithful to real nightlife behavior than a calendar-date grouping would. Assigning 00:00–05:59 back to the previous nightlife date prevents Friday-night activity after midnight from being counted as unrelated Saturday morning activity. Second, the final story communicates a clear argument instead of presenting a loose collection of charts. Third, combining static charts, interactive charts, and maps gives readers both a guided explanation and room for exploration.

There are also important limitations. Pedestrian counts are only a proxy for nightlife. Sensors count people moving through public space; they do not identify whether someone is entering a venue, leaving public transport, attending an event, or passing through the area for another reason. Sensor coverage is uneven, so the analysis is shaped by where sensors exist. The manually collected review data is useful for context, but it is less reproducible unless source URLs, collection dates, and search method are documented. Closest-sensor matching is also subjective in dense CBD areas where several sensors may be near one venue.

The scoring model is transparent but still subjective. The weights — 45% pedestrian intensity, 25% review popularity, 15% weekend lift, 10% rating, and 5% hourly peak strength — reflect an editorial decision about what should count as a strong nightlife pedestrian signal. Different weights could produce a different ranking. A future version should include a sensitivity analysis where the score is recalculated under alternative weights.

Further improvements could include a larger and better documented venue list, formal distance-based venue-to-sensor matching, event calendars, public transport proximity, weather data, holiday flags, and a sensitivity analysis for score weights. These additions would help separate venue-adjacent nightlife from other sources of late-night pedestrian movement.


## 7. Contributions

All group members contributed to the assignment, but different members took the lead on different components.

| Group member | Main responsibilities |
|---|---|
| Oliver | Data loading and cleaning, nightlife-date logic, pedestrian sensor metrics, and validation of time-window assumptions. |
| Jakob | Manual venue research, review-count/rating collection, venue-to-sensor matching, and visualization generation for the website. |
| Nicklas | Website implementation, narrative structure, explanatory copy, and integration of visual elements into the final story. |
| Oliver & Nicklas | Final explainer notebook writing, discussion section, notebook cleanup, and quality control. |
| Full group | Reviewing the final interpretation, checking caveats, and aligning the notebook with the website narrative. |


## References

City of Melbourne. (n.d.). *Pedestrian Counting System (counts per hour)*. City of Melbourne Open Data Portal. Retrieved May 12, 2026, from https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-monthly-counts-per-hour/

City of Melbourne. (n.d.). *Pedestrian Counting System - Sensor Locations*. City of Melbourne Open Data Portal. Retrieved May 12, 2026, from https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-sensor-locations/

Segel, E., & Heer, J. (2010). Narrative visualization: Telling stories with data. *IEEE Transactions on Visualization and Computer Graphics, 16*(6), 1139–1148. https://doi.org/10.1109/TVCG.2010.179

Group 19. (2026). Manually collected venue rating and review-count table, used as internal project context. Future versions should add collection date, exact source pages, and search method for each venue record.
